In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import plotly.express as px
import plotly.graph_objects as go
import utils as utils

# pd.set_option('future.no_silent_downcasting', True)


# Dataset paper cdmx

In [5]:
download_path = "../data/state_of_the_art_raw"
output_path = "../data/datasets"

# read all files in the folder

file = 'datos_unidos2.csv'
dataset_name = 'paper_cdmx'

In [102]:
data = pd.read_csv(f"{download_path}/{file}")
data.columns = [col.replace('Value_','').lower() for col in data.columns]
data.rename(columns={'date':'datetime'}, inplace=True)
data['datetime'] = pd.to_datetime(data['datetime'], format='%d-%m-%y %H:%M:%S')
data['datetime'] = data['datetime'] - pd.Timedelta(hours=1)

In [103]:
data = data.melt(
    id_vars=['datetime', 'id_station'],
    var_name='variable',
    value_name='value'
)

In [104]:
data = data.pivot(
    index=['datetime', 'variable'],
    columns='id_station',
    values='value'
).reset_index()
data.rename_axis(None, axis=1, inplace=True)


In [105]:
data['year'] = data['datetime'].dt.year
data['month'] = data['datetime'].dt.month
data['day'] = data['datetime'].dt.day
data['dow'] = data['datetime'].dt.dayofweek
data['time'] = data['datetime'].dt.hour

In [106]:
not_station_columns = ['datetime', 'year', 'month', 'day', 'time', 'dow', 'variable']
station_columns = [col for col in data.columns if col not in not_station_columns]
data[station_columns] = data[station_columns].apply(pd.to_numeric, errors='coerce')
data = data[not_station_columns + station_columns]
data.reset_index(drop=True, inplace=True)

In [ ]:

# Save the dataframe to a csv file
data.to_csv(f'{output_path}/{dataset_name}.csv', index=False)

In [111]:
bins = [0, 1, 3, 12, 24, 24*3, 24*7, 24*30, 24*30*3, 24*30*6, np.inf]
labels = np.arange(1,len(bins))
no_gap_value = 0

df_temp_gaps = utils.get_data_gaps(data, bins = bins, labels = labels, 
                                   no_gap_value = no_gap_value, format_gaps = int,
                                   datetime_column = 'datetime', variable_column = 'variable',
                                   aux_columns = ['year', 'month', 'day', 'time', 'dow'])

df_temp_gaps.to_csv(f'{output_path}/{dataset_name}_md.csv', index=False)


In [ ]:
# df.query('pollutant == "pm2"').iloc[9995:10050]
# df_consecutive_nans.query('pollutant == "pm2"').iloc[9995:10050]

# Dataset RAMA-REDMET-REDDA-REDMA (1989-2024)

## Hourly: RAMA - REDMET

In [1]:
main_folder = '../data/cdmx/'
folders = ['RAMA','REDMET']

In [4]:
df = pd.DataFrame()
for file in files:
    # Divide file name to get the date, time and contaminant
    y, m, cont = file.split("_")
    cont = cont.split(".")[0]
    
    if (int(y) >= start_year) and (int(y) <= end_year) and (cont in variables):
        
        segment = pd.read_csv(f"{download_path}/{file}", skiprows=1)
        
        # avoid unnamed columns
        segment = segment.loc[:, ~segment.columns.str.contains('^Unnamed')]
        segment['Fecha'] = pd.to_datetime(segment['Fecha'], format='%d-%m-%Y')
        
        segment['variable'] = cont
        segment['year'] = segment['Fecha'].dt.year
        segment['month'] = segment['Fecha'].dt.month
        segment['day'] = segment['Fecha'].dt.day
        segment['dow'] = segment['Fecha'].dt.dayofweek
        
        df = pd.concat([df, segment], axis=0)
        
# Create time and datetime columns
df['time'] = (df['Hora']-1).astype('int')
df['datetime'] = df['Fecha'] + pd.to_timedelta(df['time'].astype('str').str.zfill(2) + ':00:00')
df.drop(columns=['Fecha','Hora'], inplace=True)

df.replace("nr", np.nan, inplace=True)
df.reset_index(drop=True, inplace=True)
df.sort_values(by=['datetime'], inplace=True)

not_station_columns = ['datetime', 'year', 'month', 'day', 'time', 'dow', 'variable']
station_columns = [col for col in df.columns if col not in not_station_columns]
df[station_columns] = df[station_columns].apply(pd.to_numeric, errors='coerce')
df = df[not_station_columns + station_columns]
df.reset_index(drop=True, inplace=True)

# Save the dataframe to a csv file
df.to_csv(output_path + '/ds_rama2.csv', index=False)

NameError: name 'files' is not defined

In [ ]:

df = pd.DataFrame()
for folder in folders:
    print(main_folder + folder)
    files = os.listdir(main_folder + folder)
    files = [f for f in files if f.endswith('.xls')]
    
    for file in files:
        
        year = file[0:4]
        variable, _ = file[4::].lower().split('.')
        
        segment = pd.read_excel(f"{main_folder}{folder}/{file}")
        
        # avoid unnamed columns
        segment = segment.loc[:, ~segment.columns.str.contains('^Unnamed')]
        
        segment['variable'] = variable
        segment['year'] = int(year)
        
        df = pd.concat([df, segment], axis=0)
        
        
# Crea la columna fecha sumando la hora y la fecha
df['HORA'] = (df['HORA']-1).astype('str').str.zfill(2) + ':00:00'
df['datetime'] = pd.to_datetime(df['FECHA'].astype('str') + ' ' + df['HORA'], format='%Y-%m-%d %H:%M:%S')
df.drop(columns=['FECHA', 'HORA'], inplace=True)

df['time'] = df['datetime'].dt.hour.astype('int')
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['day'] = df['datetime'].dt.day
df['dow'] = df['datetime'].dt.dayofweek

df.replace("nr", np.nan, inplace=True)
df.replace(-99, np.nan, inplace = True)
df.reset_index(drop=True, inplace=True)
df.sort_values(by=['datetime'], inplace=True)
df.reset_index(drop=True, inplace=True)

not_station_columns = ['datetime', 'year', 'month', 'day', 'time', 'dow', 'variable']
station_columns = [col for col in df.columns if col not in not_station_columns]
df[station_columns] = df[station_columns].apply(pd.to_numeric, errors='coerce')
df = df[not_station_columns + station_columns]
df.reset_index(drop=True, inplace=True)

# Save the dataframe to a csv file
df.to_csv(output_path + '/ds_rama_redmet.csv', index=False)

In [15]:
bins = [0, 1, 3, 12, 24, 24*3, 24*7, 24*30, 24*30*3, 24*30*6, np.inf]
labels = np.arange(1,len(bins))
no_gap_value = 0

df_temp_gaps = utils.get_data_gaps(df, bins = bins, labels = labels, 
                                   no_gap_value = no_gap_value, format_gaps = int,
                                   datetime_column = 'datetime', variable_column = 'variable',
                                   aux_columns = ['year', 'month', 'day', 'time', 'dow'])

df_temp_gaps.to_csv(output_path + '/ds_rama_redmet_md.csv', index=False)